# SVRDD demo: run the trained system on road images

Loads the two trained detectors from the detection notebook's output (`svrdd_run`), fuses their
predictions with the frozen stage-1 settings (VGDF), computes the maintenance priority (DWPS) and draws
the result. Works on SVRDD test images (with ground truth shown side by side) or on your own photos.

Inputs to attach (Add Input): the detection notebook (its output contains `svrdd_run`), optionally the
priority notebook (`priority_run`, for the cut points). The SVRDD dataset is downloaded from Zenodo
automatically if it is not attached (Settings → Internet on). A GPU is optional.

## 1. Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=False)

In [ ]:
# Shared with the detection notebook (tag: lib).
import os, sys, json, math, time, random, hashlib, shutil, zipfile, itertools, warnings
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd

PRED_COLS = ["image_id", "class_id", "confidence", "x1", "y1", "x2", "y2"]
BOX = ["x1", "y1", "x2", "y2"]
IOU_THRS = np.round(np.arange(0.5, 0.96, 0.05), 2)
_trapz = getattr(np, "trapezoid", None) or np.trapz

def empty_preds():
    return pd.DataFrame({c: pd.Series(dtype=("object" if c == "image_id" else "int64" if c == "class_id" else "float64"))
                         for c in PRED_COLS})

def make_preds(image_ids, cls, conf, boxes):
    boxes = np.asarray(boxes, dtype=np.float64).reshape(-1, 4)
    return pd.DataFrame({"image_id": list(image_ids), "class_id": np.asarray(cls, dtype=np.int64),
                         "confidence": np.asarray(conf, dtype=np.float64),
                         "x1": boxes[:, 0], "y1": boxes[:, 1], "x2": boxes[:, 2], "y2": boxes[:, 3]})

def yolo_to_xyxy(cx, cy, w, h, W, H):
    return (cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H

def xyxy_to_yolo(x1, y1, x2, y2, W, H):
    return (x1 + x2) / 2 / W, (y1 + y2) / 2 / H, (x2 - x1) / W, (y2 - y1) / H

def box_area(b):
    b = np.asarray(b, dtype=np.float64).reshape(-1, 4)
    return np.clip(b[:, 2] - b[:, 0], 0, None) * np.clip(b[:, 3] - b[:, 1], 0, None)

def box_inter(a, b):
    a = np.asarray(a, dtype=np.float64).reshape(-1, 4); b = np.asarray(b, dtype=np.float64).reshape(-1, 4)
    lt = np.maximum(a[:, None, :2], b[None, :, :2]); rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None)
    return wh[..., 0] * wh[..., 1]

def box_iou(a, b):
    inter = box_inter(a, b)
    union = box_area(a)[:, None] + box_area(b)[None, :] - inter
    return inter / np.maximum(union, 1e-9)

# ---------------------------------------------------------------- matching and metrics
def match_predictions(pred, gt, iou_thrs=IOU_THRS):
    """Greedy detection matching, per image and class, highest confidence first.

    A prediction is a TP at threshold t if its best still-unmatched same-class GT box has IoU >= t.
    Because higher-confidence predictions are processed first, the match of a prediction does not
    depend on lower-confidence ones, so one matching serves every confidence threshold.
    Returns tp [n_pred, n_thr] (bool) and pred_gt [n_pred] (row index into gt at iou_thrs[0], or -1).
    Both are aligned with pred.reset_index(drop=True) / gt.reset_index(drop=True).
    """
    pred = pred.reset_index(drop=True); gt = gt.reset_index(drop=True)
    tp = np.zeros((len(pred), len(iou_thrs)), dtype=bool)
    pred_gt = np.full(len(pred), -1, dtype=np.int64)
    if len(pred) == 0 or len(gt) == 0:
        return tp, pred_gt
    pb, gb, conf = pred[BOX].to_numpy(float), gt[BOX].to_numpy(float), pred["confidence"].to_numpy(float)
    gt_groups = gt.groupby(["image_id", "class_id"]).indices
    for key, pidx in pred.groupby(["image_id", "class_id"]).indices.items():
        gidx = gt_groups.get(key)
        if gidx is None:
            continue
        pidx = pidx[np.argsort(-conf[pidx], kind="stable")]
        iou = box_iou(pb[pidx], gb[gidx])
        for ti, thr in enumerate(iou_thrs):
            used = np.zeros(len(gidx), dtype=bool)
            for r in range(len(pidx)):
                row = np.where(used, -1.0, iou[r])
                j = int(row.argmax())
                if row[j] >= thr:
                    used[j] = True
                    tp[pidx[r], ti] = True
                    if ti == 0:
                        pred_gt[pidx[r]] = gidx[j]
    return tp, pred_gt

def average_precision(tp_sorted, n_gt):
    """AP from TP flags sorted by descending confidence (101-point interpolation, as in Ultralytics/COCO)."""
    if n_gt == 0:
        return np.nan
    if len(tp_sorted) == 0:
        return 0.0
    tpc = np.cumsum(tp_sorted); fpc = np.cumsum(~tp_sorted)
    rec = tpc / n_gt; prec = tpc / np.maximum(tpc + fpc, 1e-9)
    mrec = np.concatenate([[0.0], rec, [1.0]]); mpre = np.concatenate([[1.0], prec, [0.0]])
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))
    x = np.linspace(0, 1, 101)
    return float(_trapz(np.interp(x, mrec, mpre), x))

def _prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp > 0 else 0.0
    r = tp / (tp + fn) if tp + fn > 0 else 0.0
    f = 2 * p * r / (p + r) if p + r > 0 else 0.0
    return p, r, f

def conf_vector(conf_thr, n_classes):
    return np.broadcast_to(np.asarray(conf_thr, dtype=np.float64), (n_classes,)).copy()

def evaluate(pred, gt, n_classes, conf_thr=0.25, class_names=None, match=None):
    """Detection P/R/F1 at an operating point (scalar or per-class confidence) plus mAP50 / mAP50-95."""
    pred = pred.reset_index(drop=True); gt = gt.reset_index(drop=True)
    tp, pred_gt = match if match is not None else match_predictions(pred, gt)
    cls = pred["class_id"].to_numpy(np.int64); conf = pred["confidence"].to_numpy(float)
    thr = conf_vector(conf_thr, n_classes)
    keep = conf >= thr[cls] if len(pred) else np.zeros(0, dtype=bool)
    gcls = gt["class_id"].to_numpy(np.int64)
    rows, TP, FP, FN = [], 0, 0, 0
    for c in range(n_classes):
        m = cls == c
        n_gt = int((gcls == c).sum())
        tpc = int((tp[m, 0] & keep[m]).sum()); fpc = int((~tp[m, 0] & keep[m]).sum()); fnc = n_gt - tpc
        p, r, f = _prf(tpc, fpc, fnc)
        order = np.argsort(-conf[m], kind="stable")
        aps = [average_precision(tp[m][order, ti], n_gt) for ti in range(tp.shape[1])]
        rows.append(dict(class_id=c, name=class_names[c] if class_names else str(c), n_gt=n_gt,
                         TP=tpc, FP=fpc, FN=fnc, P=p, R=r, F1=f, AP50=aps[0], AP50_95=float(np.mean(aps)),
                         conf_thr=float(thr[c])))
        TP += tpc; FP += fpc; FN += fnc
    per_class = pd.DataFrame(rows)
    p, r, f = _prf(TP, FP, FN)
    present = per_class[per_class.n_gt > 0]
    summary = dict(P=p, R=r, F1=f, macro_P=present.P.mean(), macro_R=present.R.mean(), macro_F1=present.F1.mean(),
                   mAP50=present.AP50.mean(), mAP50_95=present.AP50_95.mean(), TP=TP, FP=FP, FN=FN)
    summary = {k: (v if isinstance(v, int) else float(v)) for k, v in summary.items()}
    return dict(summary=summary, per_class=per_class, keep=keep, tp=tp, pred_gt=pred_gt)

def pr_curve(pred, gt, class_id, match=None):
    pred = pred.reset_index(drop=True)
    tp, _ = match if match is not None else match_predictions(pred, gt)
    m = (pred["class_id"] == class_id).to_numpy()
    order = np.argsort(-pred["confidence"].to_numpy(float)[m], kind="stable")
    t = tp[m][order, 0]
    n_gt = int((gt["class_id"] == class_id).sum())
    tpc = np.cumsum(t); fpc = np.cumsum(~t)
    return tpc / max(n_gt, 1), tpc / np.maximum(tpc + fpc, 1e-9), pred["confidence"].to_numpy(float)[m][order]

# ---------------------------------------------------------------- threshold optimisation
CONF_GRID = np.round(np.arange(0.05, 0.96, 0.01), 2)

def _counts_over_grid(conf, tp0, n_gt, grid=CONF_GRID):
    tps = np.array([(tp0 & (conf >= t)).sum() for t in grid], dtype=float)
    fps = np.array([(~tp0 & (conf >= t)).sum() for t in grid], dtype=float)
    return tps, fps, n_gt - tps

def _f1(tps, fps, fns):
    return 2 * tps / np.maximum(2 * tps + fps + fns, 1e-9)

def best_global_threshold(pred, gt, match=None, grid=CONF_GRID):
    """Single confidence threshold maximising micro F1."""
    pred = pred.reset_index(drop=True)
    tp, _ = match if match is not None else match_predictions(pred, gt, IOU_THRS[:1])
    f = _f1(*_counts_over_grid(pred["confidence"].to_numpy(float), tp[:, 0], len(gt), grid))
    i = int(f.argmax())
    return float(grid[i]), float(f[i]), f

def best_class_thresholds(pred, gt, n_classes, match=None, grid=CONF_GRID, fallback=None):
    """Per-class confidence thresholds, each maximising that class's F1 (classes without GT use fallback)."""
    pred = pred.reset_index(drop=True)
    tp, _ = match if match is not None else match_predictions(pred, gt, IOU_THRS[:1])
    if fallback is None:
        fallback = best_global_threshold(pred, gt, (tp, None), grid)[0]
    cls = pred["class_id"].to_numpy(np.int64); conf = pred["confidence"].to_numpy(float)
    gcls = gt["class_id"].to_numpy(np.int64)
    out = np.full(n_classes, fallback, dtype=float)
    for c in range(n_classes):
        n_gt = int((gcls == c).sum()); m = cls == c
        if n_gt == 0 or m.sum() == 0:
            continue
        out[c] = grid[int(_f1(*_counts_over_grid(conf[m], tp[m, 0], n_gt, grid)).argmax())]
    return out

def micro_f1(pred, gt, n_classes, conf_thr, match=None):
    pred = pred.reset_index(drop=True)
    tp, _ = match if match is not None else match_predictions(pred, gt, IOU_THRS[:1])
    thr = conf_vector(conf_thr, n_classes)
    keep = pred["confidence"].to_numpy(float) >= thr[pred["class_id"].to_numpy(np.int64)]
    TP = int((tp[:, 0] & keep).sum()); FP = int((~tp[:, 0] & keep).sum())
    return _prf(TP, FP, len(gt) - TP)[2]

def crossfit_f1(pred, gt, n_classes, mode, seed=0, folds=2):
    """Honest estimate of tuned F1: thresholds are fitted on one half of the images and scored on the other.

    mode: 'global' or 'class'. Used to decide whether per-class thresholds beat a single threshold
    without rewarding the extra degrees of freedom."""
    ids = np.array(sorted(set(gt["image_id"]) | set(pred["image_id"])))
    rng = np.random.RandomState(seed); rng.shuffle(ids)
    parts = np.array_split(ids, folds)
    scores = []
    for k in range(folds):
        test_ids = set(parts[k]); fit_ids = set(ids) - test_ids
        pf, gf = pred[pred.image_id.isin(fit_ids)], gt[gt.image_id.isin(fit_ids)]
        pt, gtt = pred[pred.image_id.isin(test_ids)], gt[gt.image_id.isin(test_ids)]
        if mode == "global":
            thr = best_global_threshold(pf, gf)[0]
        else:
            thr = best_class_thresholds(pf, gf, n_classes)
        scores.append(micro_f1(pt, gtt, n_classes, thr))
    return float(np.mean(scores))

# ---------------------------------------------------------------- fusion: WBF and NMS
def nms_numpy(boxes, scores, iou_thr):
    order = np.argsort(-scores, kind="stable"); keep = []
    while len(order):
        i = order[0]; keep.append(i)
        if len(order) == 1:
            break
        iou = box_iou(boxes[i:i + 1], boxes[order[1:]])[0]
        order = order[1:][iou <= iou_thr]
    return np.array(keep, dtype=np.int64)

def wbf_single(boxes, scores, src, weights, iou_thr=0.55, conf_type="avg"):
    """Weighted Boxes Fusion (Solovyev et al., 2021) for one image and one class.

    boxes [n,4], scores [n], src [n] = index of the model/view that produced each box.
    conf_type 'avg': fused score = mean weighted score * min(n_boxes, sum_w) / sum_w, which penalises
                     boxes that only some models found (use for model ensembles).
              'max': fused score = highest member score (use when merging views/tiles of one model,
                     where a view legitimately cannot see every object).
    """
    w = np.asarray(weights, dtype=float)
    s = scores * w[src]
    order = np.argsort(-s, kind="stable")
    clusters, fused = [], np.zeros((0, 4))
    for i in order:
        j = -1
        if len(fused):
            iou = box_iou(boxes[i:i + 1], fused)[0]
            j = int(iou.argmax())
            if iou[j] <= iou_thr:
                j = -1
        if j < 0:
            clusters.append([i]); fused = np.vstack([fused, boxes[i]])
        else:
            clusters[j].append(i)
            idx = np.array(clusters[j])
            fused[j] = (boxes[idx] * s[idx, None]).sum(0) / max(s[idx].sum(), 1e-9)
    out_s = np.zeros(len(clusters))
    for k, idx in enumerate(clusters):
        idx = np.array(idx)
        if conf_type == "max":
            out_s[k] = scores[idx].max()
        else:
            out_s[k] = s[idx].mean() * min(len(idx), w.sum()) / w.sum()
    return fused, out_s

def fuse_predictions(dfs, weights=None, method="wbf", iou_thr=0.55, conf_type="avg", skip_thr=0.0):
    """Fuse several canonical prediction tables (models or views) image by image, class by class."""
    weights = np.ones(len(dfs)) if weights is None else np.asarray(weights, dtype=float)
    parts = []
    for i, d in enumerate(dfs):
        d = d[d["confidence"] >= skip_thr][PRED_COLS].copy(); d["src"] = i; parts.append(d)
    allp = pd.concat(parts, ignore_index=True)
    if len(allp) == 0:
        return empty_preds()
    B = allp[BOX].to_numpy(float); S = allp["confidence"].to_numpy(float); SRC = allp["src"].to_numpy(np.int64)
    ids, cls, conf, boxes = [], [], [], []
    for (img, c), idx in allp.groupby(["image_id", "class_id"]).indices.items():
        if method == "wbf":
            fb, fs = wbf_single(B[idx], S[idx], SRC[idx], weights, iou_thr, conf_type)
        else:  # plain NMS on weight-scaled scores
            s = S[idx] * weights[SRC[idx]] / weights.max()
            k = nms_numpy(B[idx], s, iou_thr); fb, fs = B[idx][k], s[k]
        ids += [img] * len(fs); cls += [c] * len(fs); conf += list(fs); boxes.append(fb)
    out = make_preds(ids, cls, conf, np.vstack(boxes) if boxes else np.zeros((0, 4)))
    return out.sort_values(["image_id", "confidence"], ascending=[True, False]).reset_index(drop=True)

# ---------------------------------------------------------------- tiling
def tile_starts(length, tile, min_overlap=0.2):
    """Start offsets of tiles covering [0, length) with at least min_overlap between neighbours; last tile is flush."""
    if tile >= length:
        return [0]
    n = max(2, math.ceil((length - tile * min_overlap) / (tile * (1 - min_overlap))))
    return [int(round(i * (length - tile) / (n - 1))) for i in range(n)]

def make_tiles(W, H, tile=640, min_overlap=0.2):
    return [(x, y, min(x + tile, W), min(y + tile, H))
            for y in tile_starts(H, tile, min_overlap) for x in tile_starts(W, tile, min_overlap)]

def truncated_flags(boxes, tile, W, H, tol=2.0):
    """True where a box (already in full-image coordinates) touches an edge of its tile that is NOT an image edge.
    Such a box is probably a fragment of an object that continues in the neighbouring tile."""
    boxes = np.asarray(boxes, dtype=float).reshape(-1, 4)
    tx1, ty1, tx2, ty2 = tile
    f = np.zeros(len(boxes), dtype=bool)
    if tx1 > 0: f |= boxes[:, 0] <= tx1 + tol
    if ty1 > 0: f |= boxes[:, 1] <= ty1 + tol
    if tx2 < W: f |= boxes[:, 2] >= tx2 - tol
    if ty2 < H: f |= boxes[:, 3] >= ty2 - tol
    return f

def merge_tile_boxes(boxes, scores, tile_id, trunc, stitch_ios=0.3, contain_ios=0.6):
    """Resolve tile-boundary fragments for one image and one class (tile_id == -1 marks full-image boxes).

    1. Stitch: truncated boxes from different tiles that overlap (intersection over the smaller box
       >= stitch_ios) are fragments of one object -> replace them by their union (score = max).
    2. Contain: a still-truncated box lying mostly inside another box (>= contain_ios of its own area)
       duplicates part of that object -> drop it.
    The remaining boxes go to WBF, which merges full-view and tile detections of the same object.
    """
    boxes = np.asarray(boxes, dtype=float).reshape(-1, 4).copy(); scores = np.asarray(scores, dtype=float).copy()
    tile_id = np.asarray(tile_id).copy(); trunc = np.asarray(trunc, dtype=bool).copy()
    n = len(boxes)
    if n == 0:
        return boxes, scores, tile_id
    parent = list(range(n))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]; i = parent[i]
        return i
    t = np.where(trunc)[0]
    if len(t) > 1:
        inter = box_inter(boxes[t], boxes[t]); area = box_area(boxes[t])
        ios = inter / np.maximum(np.minimum(area[:, None], area[None, :]), 1e-9)
        for a in range(len(t)):
            for b in range(a + 1, len(t)):
                if tile_id[t[a]] != tile_id[t[b]] and ios[a, b] >= stitch_ios:
                    parent[find(t[a])] = find(t[b])
    groups = defaultdict(list)
    for i in range(n):
        groups[find(i)].append(i)
    nb, ns, nt, ntr = [], [], [], []
    for members in groups.values():
        m = np.array(members)
        if len(m) == 1:
            nb.append(boxes[m[0]]); ns.append(scores[m[0]]); nt.append(tile_id[m[0]]); ntr.append(trunc[m[0]])
        else:  # union of fragments; no longer counts as truncated
            nb.append(np.concatenate([boxes[m, :2].min(0), boxes[m, 2:].max(0)]))
            ns.append(scores[m].max()); nt.append(-2); ntr.append(False)
    boxes, scores, tile_id, trunc = np.array(nb), np.array(ns), np.array(nt), np.array(ntr, dtype=bool)
    if trunc.any() and len(boxes) > 1:
        inter = box_inter(boxes, boxes); np.fill_diagonal(inter, 0)
        frac = inter / np.maximum(box_area(boxes)[:, None], 1e-9)      # share of box i inside box j
        bigger = box_area(boxes)[None, :] > box_area(boxes)[:, None]
        drop = trunc & ((frac >= contain_ios) & bigger).any(1)
        boxes, scores, tile_id = boxes[~drop], scores[~drop], tile_id[~drop]
    return boxes, scores, tile_id

def merge_full_and_tiles(full_df, tile_df, iou_thr=0.55):
    """full_df: canonical predictions on whole images. tile_df: canonical + columns tile_id, trunc.
    Returns one canonical table without double-counted boundary objects."""
    f = full_df[PRED_COLS].copy(); f["tile_id"] = -1; f["trunc"] = False
    allp = pd.concat([f, tile_df[PRED_COLS + ["tile_id", "trunc"]]], ignore_index=True)
    if len(allp) == 0:
        return empty_preds()
    B = allp[BOX].to_numpy(float); S = allp["confidence"].to_numpy(float)
    T = allp["tile_id"].to_numpy(np.int64); TR = allp["trunc"].to_numpy(bool)
    ids, cls, conf, boxes = [], [], [], []
    for (img, c), idx in allp.groupby(["image_id", "class_id"]).indices.items():
        b, s, t = merge_tile_boxes(B[idx], S[idx], T[idx], TR[idx])
        if len(b) == 0:
            continue
        src = (t != -1).astype(np.int64)                      # 0 = full view, 1 = tiles
        fb, fs = wbf_single(b, s, src, [1.0, 1.0], iou_thr, conf_type="max")
        ids += [img] * len(fs); cls += [c] * len(fs); conf += list(fs); boxes.append(fb)
    out = make_preds(ids, cls, conf, np.vstack(boxes) if boxes else np.zeros((0, 4)))
    return out.sort_values(["image_id", "confidence"], ascending=[True, False]).reset_index(drop=True)

# ---------------------------------------------------------------- error analysis
SIZE_BINS = [("small", 0, 32 ** 2), ("medium", 32 ** 2, 96 ** 2), ("large", 96 ** 2, float("inf"))]

def size_bucket(gt):
    a = box_area(gt[BOX].to_numpy(float))
    return np.select([a < 32 ** 2, a < 96 ** 2], ["small", "medium"], default="large")

def gt_hits(res, n_gt):
    """Boolean [n_gt]: ground-truth boxes found by the kept (above-threshold) true positives of an evaluate() result."""
    hit = np.zeros(n_gt, dtype=bool)
    idx = res["pred_gt"][res["keep"] & res["tp"][:, 0]]
    hit[idx[idx >= 0]] = True
    return hit

def recall_by_size(gt, hit):
    b = size_bucket(gt)
    return pd.DataFrame([dict(size=name, n_gt=int((b == name).sum()),
                              recall=float(hit[b == name].mean()) if (b == name).any() else np.nan)
                         for name, _, _ in SIZE_BINS])

def complementarity(gt, hit_a, hit_b, class_names):
    """Which ground-truth objects does each model find? Counts sum to n_gt in every row."""
    gt = gt.reset_index(drop=True); b = size_bucket(gt); cls = gt["class_id"].to_numpy()
    def row(name, m):
        n = int(m.sum())
        both, a_only, b_only = int((hit_a & hit_b & m).sum()), int((hit_a & ~hit_b & m).sum()), int((~hit_a & hit_b & m).sum())
        neither = n - both - a_only - b_only
        return dict(group=name, n_gt=n, both=both, A_only=a_only, B_only=b_only, neither=neither,
                    recall_A=(both + a_only) / max(n, 1), recall_B=(both + b_only) / max(n, 1),
                    oracle_recall=(n - neither) / max(n, 1), B_only_share=b_only / max(n, 1))
    rows = [row("all", np.ones(len(gt), dtype=bool))]
    rows += [row(class_names[c], cls == c) for c in range(len(class_names))]
    rows += [row(name, b == name) for name, _, _ in SIZE_BINS]
    return pd.DataFrame(rows)

def shared_false_positives(pred_a, res_a, pred_b, res_b, iou_thr=0.5):
    """Share of each model's false positives that the other model also produces (same class, IoU >= thr)."""
    fa = pred_a.reset_index(drop=True)[res_a["keep"] & ~res_a["tp"][:, 0]]
    fb = pred_b.reset_index(drop=True)[res_b["keep"] & ~res_b["tp"][:, 0]]
    gb = fb.groupby(["image_id", "class_id"]).indices
    Bb = fb[BOX].to_numpy(float); Ba = fa[BOX].to_numpy(float)
    shared_a = 0; hit_b = np.zeros(len(fb), dtype=bool)
    for key, ia in fa.groupby(["image_id", "class_id"]).indices.items():
        ib = gb.get(key)
        if ib is None:
            continue
        iou = box_iou(Ba[ia], Bb[ib])
        shared_a += int((iou.max(1) >= iou_thr).sum()); hit_b[ib[iou.max(0) >= iou_thr]] = True
    return dict(fp_A=len(fa), fp_B=len(fb), fp_A_shared=shared_a, fp_B_shared=int(hit_b.sum()),
                A_shared_frac=shared_a / max(len(fa), 1), B_shared_frac=float(hit_b.sum()) / max(len(fb), 1))

def detection_confusion(pred, gt, n_classes, conf_thr, iou_thr=0.5):
    """Confusion matrix with a background row/column: rows = true class, columns = predicted class.
    Matching here ignores class (best IoU >= thr), so off-diagonal cells are localised-but-misclassified boxes."""
    thr = conf_vector(conf_thr, n_classes)
    p = pred[pred["confidence"].to_numpy(float) >= thr[pred["class_id"].to_numpy(np.int64)]]
    M = np.zeros((n_classes + 1, n_classes + 1), dtype=np.int64)
    gg = gt.groupby("image_id").indices; pg = p.groupby("image_id").indices
    GB, GC = gt[BOX].to_numpy(float), gt["class_id"].to_numpy(np.int64)
    PB, PC, PS = p[BOX].to_numpy(float), p["class_id"].to_numpy(np.int64), p["confidence"].to_numpy(float)
    for img in set(gg) | set(pg):
        gi = gg.get(img, np.array([], dtype=np.int64)); pi = pg.get(img, np.array([], dtype=np.int64))
        used = np.zeros(len(gi), dtype=bool)
        if len(gi) and len(pi):
            pi = pi[np.argsort(-PS[pi], kind="stable")]
            iou = box_iou(PB[pi], GB[gi])
            # prefer a same-class partner, otherwise the best-overlapping box of any class
            for r in range(len(pi)):
                row = np.where(used, -1.0, iou[r])
                same = np.where(GC[gi] == PC[pi[r]], row, -1.0)
                j = int(same.argmax()) if same.max() >= iou_thr else int(row.argmax())
                if row[j] >= iou_thr:
                    used[j] = True; M[GC[gi[j]], PC[pi[r]]] += 1
                else:
                    M[n_classes, PC[pi[r]]] += 1
        else:
            for r in pi:
                M[n_classes, PC[r]] += 1
        for j in np.where(~used)[0]:
            M[GC[gi[j]], n_classes] += 1
    return M

In [ ]:
import re, glob
import torch
from PIL import Image
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

In [ ]:
# Shared with the detection notebook (tag: prilib).
def box_contributions(df, wh, weight_vec, linear_mask, base):
    """Contribution of every box to its image's priority score: weight[class] * (base + extent).

    extent is scale-free and in (0, 1]: for linear damage the box diagonal over the image diagonal (a length),
    for areal damage the square root of the box area over the image area. wh maps image_id -> (W, H)."""
    if len(df) == 0:
        return np.zeros(0)
    W = df["image_id"].map(lambda i: wh[i][0]).to_numpy(float); H = df["image_id"].map(lambda i: wh[i][1]).to_numpy(float)
    w = (df["x2"] - df["x1"]).to_numpy(float).clip(0); h = (df["y2"] - df["y1"]).to_numpy(float).clip(0)
    cls = df["class_id"].to_numpy(np.int64)
    length = np.hypot(w, h) / np.hypot(W, H)
    areal = np.sqrt(w * h / (W * H))
    extent = np.where(np.asarray(linear_mask)[cls], length, areal)
    return np.asarray(weight_vec, dtype=float)[cls] * (base + extent)

def image_scores(df, image_ids, wh, weight_vec, linear_mask, base):
    """Priority score per image (sum of box contributions); images without boxes score 0."""
    s = pd.Series(box_contributions(df, wh, weight_vec, linear_mask, base), index=df["image_id"].to_numpy())
    return s.groupby(level=0).sum().reindex(list(image_ids), fill_value=0.0).to_numpy(float)

def to_levels(scores, cuts):
    """0 = Low, 1 = Medium, 2 = High. A score equal to a cut point belongs to the higher level."""
    return np.digitize(np.asarray(scores, dtype=float), np.asarray(cuts, dtype=float), right=False)

def level_metrics(y_true, y_pred):
    return dict(accuracy=accuracy_score(y_true, y_pred),
                macro_F1=f1_score(y_true, y_pred, average="macro", labels=[0, 1, 2], zero_division=0),
                QWK=cohen_kappa_score(y_true, y_pred, weights="quadratic", labels=[0, 1, 2]),
                high_recall=float(np.mean(np.asarray(y_pred)[np.asarray(y_true) == 2] == 2)) if (np.asarray(y_true) == 2).any() else np.nan)

def score_metrics(s_true, s_pred):
    rho = spearmanr(s_true, s_pred).statistic if len(s_true) > 2 and np.std(s_pred) > 0 and np.std(s_true) > 0 else np.nan
    return dict(spearman=float(rho), MAE=float(np.mean(np.abs(np.asarray(s_true) - np.asarray(s_pred)))))

_XY = re.compile(r"_(\d{8,11})_(\d{8,11})$")
def parse_xy(image_id):
    """Map coordinates in metres from an SVRDD file name '<panorama id>_<x*100>_<y*100>', or (nan, nan)."""
    m = _XY.search(str(image_id))
    return (int(m.group(1)) / 100.0, int(m.group(2)) / 100.0) if m else (np.nan, np.nan)

def segment_ids(x, y, cell):
    """Grid-cell id per point; '' where the coordinates are missing."""
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    gx = np.floor(np.where(ok, x, 0) / cell).astype(np.int64); gy = np.floor(np.where(ok, y, 0) / cell).astype(np.int64)
    return np.where(ok, np.char.add(np.char.add(gx.astype(str), "_"), gy.astype(str)), "")

def top_k_overlap(a, b, k):
    """Share of the k highest-ranked items under a that are also among the k highest under b."""
    k = int(max(1, min(k, len(a))))
    ia = set(np.argsort(-np.asarray(a), kind="stable")[:k]); ib = set(np.argsort(-np.asarray(b), kind="stable")[:k])
    return len(ia & ib) / k

## 2. Configuration

In [ ]:
INPUT = Path(os.environ.get("DEMO_INPUT", "/kaggle/input"))
MY_IMAGES = os.environ.get("DEMO_MY_IMAGES")     # optional folder with your own road photos
N_TEST_IMAGES = 6                                # random SVRDD test images to show
SEED = 7
DOWNLOAD_IF_MISSING = True                       # fetch SVRDD_YOLO.zip (2.9 GB) from Zenodo if the dataset is not attached
ZENODO_URL = "https://zenodo.org/api/records/10100129/files/SVRDD_YOLO.zip/content"
DATA_DIR = Path(os.environ.get("DEMO_DATA", "/kaggle/working/svrdd_data"))
CLASS_NAMES = ["longitudinal_crack", "transverse_crack", "alligator_crack", "pothole",
               "manhole_cover", "longitudinal_patch", "transverse_patch"]
CLASS_WEIGHT = {"pothole": 10, "alligator_crack": 8, "transverse_crack": 5, "longitudinal_crack": 4,
                "longitudinal_patch": 2, "transverse_patch": 2, "manhole_cover": 0}
LINEAR_CLASSES = {"longitudinal_crack", "transverse_crack", "longitudinal_patch", "transverse_patch"}
BASE = 0.1
DEFAULT_CUTS = [2.0228, 3.7917]                 # cut points measured in the priority notebook
LEVELS = ["Low", "Medium", "High"]
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7"]
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
NC = len(CLASS_NAMES)
W_VEC = np.array([CLASS_WEIGHT[n] for n in CLASS_NAMES], dtype=float)
LIN = np.array([n in LINEAR_CLASSES for n in CLASS_NAMES])

def find_dir(name, must_have):
    hits = sorted((p for p in INPUT.rglob(name) if p.is_dir() and (p / must_have).exists()), key=lambda p: len(p.parts))
    return hits[0] if hits else None

STAGE1 = Path(os.environ["DEMO_STAGE1"]) if os.environ.get("DEMO_STAGE1") else find_dir("svrdd_run", "state.json")
assert STAGE1 is not None, "Attach the detection notebook's output (it contains svrdd_run)."
S1 = json.loads((STAGE1 / "state.json").read_text())
A_W = STAGE1 / "runs" / "A_yolo11m" / "weights" / "best.pt"
B_W = STAGE1 / "runs" / "B_frcnn_r50" / "best.pt"
pr = find_dir("priority_run", "summary.json")
CUTS = json.loads((pr / "summary.json").read_text())["cut_points"] if pr else DEFAULT_CUTS
print("stage 1:", STAGE1, "| final detector:", "+".join(S1["FINAL_KEYS"]), "| threshold:", S1["FINAL_thr"], "| cut points:", np.round(CUTS, 3))
print("YOLO11m weights:", A_W.exists(), "| Faster R-CNN weights:", B_W.exists(), "| device:", DEVICE)

## 3. Load the trained detectors

In [ ]:
from ultralytics import YOLO
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

yolo = YOLO(str(A_W))
frcnn = None
if "B" in S1["FINAL_KEYS"]:
    frcnn = fasterrcnn_resnet50_fpn(weights=None, weights_backbone=None, min_size=1024, max_size=1024,
                                    box_score_thresh=0.001, box_detections_per_img=300, box_nms_thresh=0.5)
    frcnn.roi_heads.box_predictor = FastRCNNPredictor(frcnn.roi_heads.box_predictor.cls_score.in_features, NC + 1)
    sd = torch.load(B_W, map_location="cpu"); frcnn.load_state_dict(sd.get("model", sd)); frcnn.to(DEVICE).eval()
print("loaded:", "YOLO11m" + (" + Faster R-CNN" if frcnn is not None else ""))

def predict_yolo(img_rgb, view):
    arr = img_rgb[:, ::-1] if view.get("flip") else img_rgb
    res = yolo.predict([np.ascontiguousarray(arr[:, :, ::-1])], imgsz=view["imgsz"], conf=0.001, iou=view.get("nms_iou", 0.6),
                       max_det=300, verbose=False, device=0 if torch.cuda.is_available() else "cpu")[0]
    b = res.boxes.xyxy.float().cpu().numpy().astype(float); s = res.boxes.conf.float().cpu().numpy(); c = res.boxes.cls.cpu().numpy().astype(int)
    if view.get("flip"):
        b[:, [0, 2]] = img_rgb.shape[1] - b[:, [2, 0]]
    return b, s, c

@torch.no_grad()
def predict_frcnn(img_rgb, view):
    arr = img_rgb[:, ::-1] if view.get("flip") else img_rgb
    frcnn.transform.min_size = (view["imgsz"],); frcnn.transform.max_size = view["imgsz"]
    x = torch.from_numpy(np.ascontiguousarray(arr)).permute(2, 0, 1).float().div(255).to(DEVICE)
    o = frcnn([x])[0]
    b = o["boxes"].float().cpu().numpy().astype(float); s = o["scores"].float().cpu().numpy(); c = o["labels"].cpu().numpy().astype(int) - 1
    if view.get("flip"):
        b[:, [0, 2]] = img_rgb.shape[1] - b[:, [2, 0]]
    return b, s, c

def model_preds(key, image_id, img):
    fn = predict_yolo if key == "A" else predict_frcnn
    dfs = [make_preds([image_id] * len(s), c, s, b) for b, s, c in (fn(img, v) for v in S1[f"{key}_views"])]
    return dfs[0] if len(dfs) == 1 else fuse_predictions(dfs, method="wbf", iou_thr=0.55, conf_type="max")

def analyse(path):
    """Fused detections above the frozen threshold, priority score and level for one image."""
    image_id = Path(path).stem
    with Image.open(path) as im:
        img = np.asarray(im.convert("RGB"))
    t0 = time.time()
    preds = {k: model_preds(k, image_id, img) for k in S1["FINAL_KEYS"]}
    if S1["FINAL_KEYS"] == ["A"]:
        d = preds["A"]
    else:
        F = S1["FUSE"]; w = [F["weights"][["A", "B", "C"].index(k)] for k in S1["FINAL_KEYS"]]
        d = fuse_predictions([preds[k] for k in S1["FINAL_KEYS"]], w, F["method"], F["iou"], F["conf_type"], skip_thr=F.get("skip", 0.05))
    thr = conf_vector(S1["FINAL_thr"], NC)
    d = d[d.confidence.to_numpy() >= thr[d.class_id.to_numpy()]].reset_index(drop=True)
    wh = {image_id: (img.shape[1], img.shape[0])}
    score = float(image_scores(d, [image_id], wh, W_VEC, LIN, BASE)[0])
    level = LEVELS[int(to_levels([score], CUTS)[0])]
    return img, d, score, level, time.time() - t0

def draw(ax, img, boxes, title):
    ax.imshow(img); ax.set_title(title, fontsize=9); ax.axis("off")
    for r_ in boxes.itertuples():
        col = COLORS[int(r_.class_id) % len(COLORS)]
        ax.add_patch(Rectangle((r_.x1, r_.y1), r_.x2 - r_.x1, r_.y2 - r_.y1, fill=False, edgecolor=col, linewidth=1.6))
        label = CLASS_NAMES[int(r_.class_id)].replace("_", " ") + (f" {r_.confidence:.2f}" if hasattr(r_, "confidence") else "")
        ax.text(r_.x1, max(r_.y1 - 4, 8), label, fontsize=6, color="white", bbox=dict(facecolor=col, edgecolor="none", pad=1))

## 4. SVRDD test images: prediction next to ground truth

In [ ]:
import zipfile
def find_split_file(base):
    return next((p for p in base.rglob("test.txt") if p.parent.name.lower().startswith("svrdd")), None) if base.exists() else None

data_root = find_split_file(INPUT) or find_split_file(DATA_DIR)
if data_root is None and DOWNLOAD_IF_MISSING:
    DATA_DIR.mkdir(parents=True, exist_ok=True); z = DATA_DIR / "SVRDD_YOLO.zip"
    print("downloading SVRDD_YOLO.zip from Zenodo (2.9 GB; notebook internet must be ON, a few minutes on Kaggle) ...")
    subprocess.run(["wget", "-q", "-c", "-O", str(z), ZENODO_URL], check=True)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(DATA_DIR)
    z.unlink()
    data_root = find_split_file(DATA_DIR)
if data_root is None:
    print("SVRDD not attached: skip this section, or attach the dataset to compare with ground truth.")
else:
    root = data_root.parent
    stems = [Path(l.strip().replace("\\", "/")).stem for l in data_root.read_text().splitlines() if l.strip()]
    imgs = {p.stem: p for p in root.rglob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")}
    labels = {p.stem: p for p in root.rglob("*.txt") if p.stem in imgs}
    chosen = [s for s in np.random.RandomState(SEED).permutation(stems) if s in imgs][:N_TEST_IMAGES]
    rows = []
    fig, axes = plt.subplots(len(chosen), 2, figsize=(10, 5 * len(chosen)), squeeze=False)
    for k, s in enumerate(chosen):
        img, d, score, level, secs = analyse(imgs[s])
        H, W = img.shape[:2]; gt = []
        for line in (labels[s].read_text().splitlines() if s in labels else []):
            v = line.split()
            if len(v) >= 5:
                c, cx, cy, w, h = int(float(v[0])), *map(float, v[1:5])
                gt.append((c,) + tuple(yolo_to_xyxy(cx, cy, w, h, W, H)))
        gt = pd.DataFrame(gt, columns=["class_id", "x1", "y1", "x2", "y2"])
        ref = float(image_scores(gt.assign(image_id=s), [s], {s: (W, H)}, W_VEC, LIN, BASE)[0])
        draw(axes[k, 0], img, d, f"Prediction: {len(d)} damages | priority {level} (score {score:.2f}) | {secs:.2f} s")
        draw(axes[k, 1], img, gt, f"Ground truth: {len(gt)} damages | priority {LEVELS[int(to_levels([ref], CUTS)[0])]} (score {ref:.2f})")
        rows.append(dict(image=s[:24], predicted=len(d), ground_truth=len(gt), pred_level=level,
                         ref_level=LEVELS[int(to_levels([ref], CUTS)[0])], seconds=round(secs, 2)))
    plt.tight_layout(); plt.show()
    print(pd.DataFrame(rows).to_string(index=False))

## 5. Your own road photos (optional)
Upload photos as a Kaggle dataset, attach it, and set `MY_IMAGES` in section 2 to its folder.

In [ ]:
if MY_IMAGES and Path(MY_IMAGES).exists():
    paths = sorted(p for p in Path(MY_IMAGES).rglob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png"))[:8]
    fig, axes = plt.subplots(len(paths), 1, figsize=(8, 6 * len(paths)), squeeze=False)
    for k, pth in enumerate(paths):
        img, d, score, level, secs = analyse(pth)
        draw(axes[k, 0], img, d, f"{pth.name}: {len(d)} damages | priority {level} (score {score:.2f})")
        print(pth.name, "->", level, f"score {score:.2f}", "|", d.class_id.map(lambda c: CLASS_NAMES[c]).value_counts().to_dict())
    plt.tight_layout(); plt.show()
else:
    print("No folder of own photos set (MY_IMAGES).")